#Kaggle to S3
> Send all the historic data from kaggle to s3

In [0]:
%pip install kaggle boto3

In [0]:
%run /Repos/nikum.vedansh@gmail.com/formula-one-project/configs/credentials

In [0]:
os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN
kaggle.api.authenticate()

In [0]:
import io
import zipfile
import requests
import boto3
import kaggle
import time
from datetime import datetime

In [0]:
s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)

In [0]:
succeeded, failed = [], []
kaggle_auth = (KAGGLE_USERNAME, KAGGLE_KEY)

In [0]:
start_time = time.time()
print("=" * 40)
print(f"  Kaggle → S3 Upload Started")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 40 + "\n")

for dataset_id, folder in DATASETS.items():
    print(f"📦 Streaming {dataset_id}...")
    
    try:
        # 1. Download the zip directly into RAM
        # We bypass the 'download_files' method to get raw bytes
        url = f"https://www.kaggle.com/api/v1/datasets/download/{dataset_id}"
        response = requests.get(url, auth=kaggle_auth)
        
        if response.status_code != 200:
            raise Exception(f"Kaggle download failed with status {response.status_code}")

        # 2. Treat the bytes as a virtual file (BytesIO)
        zip_buffer = io.BytesIO(response.content)

        # 3. Open the zip in-memory
        with zipfile.ZipFile(zip_buffer) as z:
            for filename in z.namelist():
                # Filter for CSVs and ignore internal metadata/folders
                if filename.endswith('.csv') and not filename.startswith('__MACOSX'):
                    try:
                        # 4. Read the file content and push to S3
                        with z.open(filename) as f:
                            csv_content = f.read()
                            
                            s3.put_object(
                                Bucket=BUCKET,
                                Key=f"{folder}/{filename}",
                                Body=csv_content
                            )
                            
                        print(f"  ✅ {filename}")
                        succeeded.append(filename)
                    except Exception as inner_e:
                        print(f"  ❌ Error uploading {filename}: {inner_e}")
                        failed.append((filename, str(inner_e)))

    except Exception as e:
        print(f"  ❌ Dataset Failure {dataset_id}: {e}")
        failed.append((dataset_id, str(e)))

total_duration = round(time.time() - start_time, 2)

print("\n" + "=" * 40)
print(f"  Finished — {total_duration}s")
print("=" * 40)

print(f"\n  ✅ Succeeded ({len(succeeded)})")
for name in succeeded:
    print(f"     - {name}")

print(f"\n  ❌ Failed ({len(failed)})")
if failed:
    for name, reason in failed:
        print(f"     - {name}")
        print(f"       {reason}")
else:
    print("     None")

print("\n" + "=" * 40)